In [1]:
import numpy as np
import astropy.units as u
from astropy.io import fits
from astropy.wcs import WCS
from astropy.coordinates import SkyCoord, FK5
import reproject as rp

# ==================================================
# 1. FK5 rectangle mask on GAL-CAR grid
# ==================================================
def fk5_rectangle_mask_on_galcar(
    data,
    header,
    ra_min, ra_max,
    dec_min, dec_max
):
    wcs = WCS(header)
    ny, nx = data.shape

    y, x = np.mgrid[0:ny, 0:nx]
    pix_x = x + 1.0
    pix_y = y + 1.0

    lon, lat = wcs.pixel_to_world_values(pix_x, pix_y)

    gal = SkyCoord(
        l=lon * u.deg,
        b=lat * u.deg,
        frame="galactic"
    )

    fk5 = gal.transform_to(FK5(equinox="J2000"))

    ra = fk5.ra.wrap_at(360 * u.deg).deg
    dec = fk5.dec.deg

    mask = (
        (ra >= ra_min) & (ra < ra_max) &
        (dec >= dec_min) & (dec < dec_max)
    )

    return mask

In [2]:
# ==================================================
# 2. Construct FK5–CAR WCS
# ==================================================
def make_fk5_car_wcs(
    ra_min, ra_max,
    dec_min, dec_max,
    cdelt
):
    nx = int(np.round((ra_max - ra_min) / abs(cdelt)))
    ny = int(np.round((dec_max - dec_min) / abs(cdelt)))

    w = WCS(naxis=2)

    w.wcs.ctype = ["RA---CAR", "DEC--CAR"]
    w.wcs.cunit = ["deg", "deg"]
    w.wcs.cdelt = [-abs(cdelt), abs(cdelt)]

    w.wcs.crval = [
        0.5 * (ra_min + ra_max),
        0.5 * (dec_min + dec_max)
    ]
    w.wcs.crpix = [
        nx / 2 + 0.5,
        ny / 2 + 0.5
    ]

    if w.wcs.crval[1] != 0.0:
        w.wcs.crpix[1] -= w.wcs.crval[1] / w.wcs.cdelt[1]
        w.wcs.crval[1] = 0.0

    w.wcs.radesys = "FK5"
    w.wcs.equinox = 2000.0

    return w, nx, ny

In [3]:
# ==================================================
# 3. Main pipeline (moment 0)
# ==================================================
infile = "hi4pi-hvc-nhi-gal-car.fits"

ra_min, ra_max   = 60.0, 80.0
dec_min, dec_max = -13.0, 2.0

cdelt = 1.0 / 12.0  # HI4PI angular resolution (deg)

# ---------- read original GAL-CAR ----------
hdu = fits.open(infile)[0]
data = hdu.data.astype(float)
header = hdu.header

# ---------- data from log(NHI) to K km/s ----------
data = 10**data / 1.823e18  # K km/s

# ---------- FK5 mask (exact geometry) ----------
buffer_deg = 1.0

mask = fk5_rectangle_mask_on_galcar(
    data, header,
    ra_min - buffer_deg,
    ra_max + buffer_deg,
    dec_min - buffer_deg,
    dec_max + buffer_deg
)

data[~mask] = np.nan

# ---------- FK5–CAR WCS ----------
wcs_fk5, nx, ny = make_fk5_car_wcs(
    ra_min, ra_max,
    dec_min, dec_max,
    cdelt
)

# ---------- reproject ----------
array_fk5, footprint = rp.reproject_interp(
    (data, header),
    wcs_fk5,
    shape_out=(ny, nx),
)

# ---------- final header ----------
hdr_out = wcs_fk5.to_header()
hdr_out["BUNIT"] = "K km / s"
hdr_out["OBJECT"] = "HI4PI"
hdr_out["TELESCOP"] = header.get("TELESCOP", "")

# ---------- write final FITS ----------
outfile = "HI4PI_RA60_80_DEC-13_2_HVC_moment_0.fits"

fits.PrimaryHDU(
    array_fk5.astype(np.float32),
    header=hdr_out
).writeto(outfile, overwrite=True)

print(f"Saved: {outfile}")


Saved: HI4PI_RA60_80_DEC-13_2_HVC_moment_0.fits


In [4]:
# ==================================================
# 4. Main pipeline (moment 1)
# ==================================================
infile = "hi4pi-hvc-vlsr-gal-car.fits"

ra_min, ra_max   = 60.0, 80.0
dec_min, dec_max = -13.0, 2.0

cdelt = 1.0 / 12.0  # HI4PI angular resolution (deg)

# ---------- read original GAL-CAR ----------
hdu = fits.open(infile)[0]
data = hdu.data.astype(float)
header = hdu.header

# ---------- FK5 mask (exact geometry) ----------
buffer_deg = 1.0

mask = fk5_rectangle_mask_on_galcar(
    data, header,
    ra_min - buffer_deg,
    ra_max + buffer_deg,
    dec_min - buffer_deg,
    dec_max + buffer_deg
)

data[~mask] = np.nan

# ---------- FK5–CAR WCS ----------
wcs_fk5, nx, ny = make_fk5_car_wcs(
    ra_min, ra_max,
    dec_min, dec_max,
    cdelt
)

# ---------- reproject ----------
array_fk5, footprint = rp.reproject_interp(
    (data, header),
    wcs_fk5,
    shape_out=(ny, nx),
)

# ---------- final header ----------
hdr_out = wcs_fk5.to_header()
hdr_out["BUNIT"] = header.get("BUNIT", "")
hdr_out["OBJECT"] = "HI4PI"
hdr_out["TELESCOP"] = header.get("TELESCOP", "")

# ---------- write final FITS ----------
outfile = "HI4PI_RA60_80_DEC-13_2_HVC_moment_1.fits"

fits.PrimaryHDU(
    array_fk5.astype(np.float32),
    header=hdr_out
).writeto(outfile, overwrite=True)

print(f"Saved: {outfile}")


Saved: HI4PI_RA60_80_DEC-13_2_HVC_moment_1.fits


In [8]:
# ==================================================
# 4. Main pipeline (moment 1)
# ==================================================
infile = "hi4pi-hvc-vlsr-gal-car.fits"

ra_min, ra_max   = 14.0, 22.0
dec_min, dec_max = 25.0, 32.0

cdelt = 1.0 / 12.0  # HI4PI angular resolution (deg)

# ---------- read original GAL-CAR ----------
hdu = fits.open(infile)[0]
data = hdu.data.astype(float)
header = hdu.header

# ---------- FK5 mask (exact geometry) ----------
buffer_deg = 1.0

mask = fk5_rectangle_mask_on_galcar(
    data, header,
    ra_min - buffer_deg,
    ra_max + buffer_deg,
    dec_min - buffer_deg,
    dec_max + buffer_deg
)

data[~mask] = np.nan

# ---------- FK5–CAR WCS ----------
wcs_fk5, nx, ny = make_fk5_car_wcs(
    ra_min, ra_max,
    dec_min, dec_max,
    cdelt
)

# ---------- reproject ----------
array_fk5, footprint = rp.reproject_interp(
    (data, header),
    wcs_fk5,
    shape_out=(ny, nx),
)

# ---------- final header ----------
hdr_out = wcs_fk5.to_header()
hdr_out["BUNIT"] = header.get("BUNIT", "")
hdr_out["OBJECT"] = "HI4PI"
hdr_out["TELESCOP"] = header.get("TELESCOP", "")

# ---------- write final FITS ----------
outfile = "HI4PI_RA14_22_DEC25_32_HVC_moment_1.fits"

fits.PrimaryHDU(
    array_fk5.astype(np.float32),
    header=hdr_out
).writeto(outfile, overwrite=True)

print(f"Saved: {outfile}")


Saved: HI4PI_RA14_22_DEC25_32_HVC_moment_1.fits
